# 1. 패키지 설치

In [ ]:
# uv add pypdf=">=4.2.0,<5.0.0"
# uv add langchain-upstage

# 2. 환경변수 불러오기

- `.env` 파일에 `UPSTAGE_API_KEY` 등록

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
#print(UPSTAGE_API_KEY[28:])

#### Upstage 제공 모델 목록 조회

- Upstage API는 OpenAI 호환이므로 `GET /v1/models`로 사용 가능한 모델 목록을 조회할 수 있음
- 조회된 `id`를 `ChatUpstage(model=...)`에 그대로 사용
- Gemini와 달리 기능 구분 필드(`supported_actions`)가 없어서 채팅용 모델만 필터링할 수는 없음

In [2]:
import requests

UPSTAGE_BASE_URL = "https://api.upstage.ai/v1"

# Upstage 모델 목록 조회 (OpenAI 호환 /models 엔드포인트)
resp = requests.get(
    f"{UPSTAGE_BASE_URL}/models",
    headers={"Authorization": f"Bearer {UPSTAGE_API_KEY}"},
    timeout=30,
)
resp.raise_for_status()  # 인증 실패 등 HTTP 에러 시 예외 발생

# 최신 모델이 위로 오도록 생성일(created) 내림차순 정렬
for m in sorted(resp.json()["data"], key=lambda m: m["created"], reverse=True):
    # 이 id를 ChatUpstage(model=...)에 그대로 사용 가능
    print(m["id"], "|", m["created"])

solar-decide-flash | 20260929
solar-decide | 20260928
solar-mini4-260922 | 20260922
solar-jev | 20260922
solar-mini4 | 20260906
solar-pro4 | 20260806
solar-pro4-260806 | 20260806
solar-pro3-260323 | 20260323
solar-pro3 | 20260126
solar-pro2-251215 | 20251215
syn-pro | 20251021
syn-pro-251021 | 20251021
solar-pro2 | 20250710
solar-mini-250422 | 20250422
solar-mini | 20240612


# 3. LLM 답변 생성

- Upstage Console에서 발급받은 API Key를 `UPSTAGE_API_KEY`라고 저장하면 별도의 설정 없이 `ChatUpstage`를 사용할 수 있음

In [3]:
from langchain_upstage import ChatUpstage

UPSTAGE_MODEL = "solar-pro3"

#llm = ChatUpstage(temperature=0.5)
llm = ChatUpstage(
        model=UPSTAGE_MODEL,
        base_url=UPSTAGE_BASE_URL,
        temperature=0.5
)
print(llm.model_name)

solar-pro3


In [4]:
ai_message=llm.invoke("LangChain은 무엇인가요?")
print(type(ai_message))
print(ai_message.content)

<class 'langchain_core.messages.ai.AIMessage'>
**LangChain**은 자연어 처리(NLP)와 대규모 언어 모델(LLM)을 활용해 **애플리케이션을 쉽고 일관되게 구축**할 수 있게 도와주는 **오픈소스 프레임워크**입니다.  
2022년 ~ 2023년 사이에 처음 공개됐으며, 이후 빠르게 성장하고 있는 커뮤니티와 함께 다양한 플러그인과 예제가 제공되고 있습니다.

---

## 핵심 개념

| 개념 | 설명 |
|------|------|
| **Chain** | 여러 단계(프롬프트, 모델 호출, 파싱, 출력 변환 등)를 순차적으로 연결해 하나의 파이프라인을 만든 단위. `SequentialChain`, `ParallelChain`, `MapReduceChain` 등 다양한 형태가 있습니다. |
| **Prompt** | LLM에 전달할 질문·지시문. `PromptTemplate`을 사용해 동적으로 변수 삽입이 가능합니다. |
| **Memory** | 대화·세션 상태를 저장·조회하는 기능. `ConversationBufferMemory`, `SummaryMemory`, `VectorStore-backed Memory` 등 여러 구현이 있습니다. |
| **Agents** | LLM이 스스로 **도구(Tool)** 를 선택하고 호출해 문제를 해결하도록 하는 에이전트. 예: `ChatOpenAI` + `function-calling` + `Tool` 정의. |
| **Tools / LLMs** | 외부 API, 로컬 모델, 함수 호출 등을 추상화한 인터페이스. OpenAI, Anthropic, Cohere, HuggingFace, Google Vertex 등 다양한 공급자를 지원합니다. |
| **Vector Stores** | 임베딩을 저장하고 유사도 검색을 수행하는 저장소. `FAISS`, `Pinecone`, `Weaviate`, `Chroma` 등과 연동됩니다. |
| **Callbacks / Observabili

In [ ]:
# using chat stream
for chunk in llm.stream("LangChain은 무엇인가요?"):
    print(chunk.content, end="")

#### ChatPromptTemplate 과 ChatUpstage 사용

In [ ]:
from langchain_upstage import ChatUpstage
from langchain_core.prompts import ChatPromptTemplate

translation_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a professional translator specializing in Korean-English translation."),
        ("human", "Translate this from {source_lang} to {target_lang}: {text}")
    ])

print(llm.model_name)

# 체인 실행
chain = translation_prompt | llm

response = chain.invoke({
    "source_lang": "English",
    "target_lang": "Korean", 
    "text": "LangChain is a powerful framework for building AI applications."
})

print("Upstage Response:")
print(response.content)
    

#### 상황·말투를 변수로 받는 번역 프롬프트

위 예제는 **언어 방향**(`source_lang`, `target_lang`)을 변수로 받았습니다. 아래 예제는 같은 문장(`Hello, How are you?`)이라도 **누구에게, 어떤 상황에서** 말하느냐에 따라 번역이 달라지도록 `situation`(상황)과 `tone`(말투)을 변수로 받습니다.

- **프롬프트 설계**: system 메시지에 역할("한국어 현지화 전문가")과 출력 규칙("번역문만 출력")을 두고, human 메시지에 상황·말투·원문을 구조화해 전달합니다.
- **`StrOutputParser`**: `response.content` 대신 문자열을 바로 받습니다.
- **`chain.batch()`**: 같은 체인에 입력 3개를 한 번에 보내 결과를 비교합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# llm 은 위에서 만든 ChatUpstage 를 그대로 사용합니다.

# 상황(situation)과 말투(tone)를 변수로 받는 번역 프롬프트: 같은 문장도 관계에 맞게 다르게 번역
tone_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a Korean localization expert. "
                   "Adapt the English phrase into natural Korean that fits the speaker's relationship and situation. "
                   "Output only the Korean translation, with no explanation."),
        ("human", "Situation: {situation}\nTone: {tone}\nEnglish: {english_text}"),
    ]
)

# 체인 구성: 프롬프트 → 모델 → 문자열 파서 (response.content 대신 문자열을 바로 반환)
print(llm.model_name)
chain = tone_prompt | llm | StrOutputParser()

english_text = "Hello, How are you?"

# 같은 문장을 세 가지 상황으로 한 번에 요청 (batch)
cases = [
    {"situation": "거래처 담당자에게 보내는 비즈니스 이메일의 첫 인사", "tone": "격식체(하십시오체)"},
    {"situation": "오랜만에 만난 대학 동기에게 건네는 인사", "tone": "친근한 반말"},
    {"situation": "명절에 찾아뵌 할아버지께 드리는 인사", "tone": "공손한 존댓말(해요체)"},
]

results = chain.batch([{**case, "english_text": english_text} for case in cases])

print(f"원문: {english_text}\n")
for case, result in zip(cases, results):
    print(f"[{case['tone']}] {case['situation']}")
    print(f"  → {result}\n")

#### 프롬프트에 기준을 명시하면 출력이 통제된다 — 비교 실험

번역을 "그냥 번역해 줘"라고만 요청하면 모델은 자기 판단대로 번역합니다. 번역 기준(**용어, 말투, 길이**)을 프롬프트에 **명시**하면 출력이 얼마나 통제되는지 두 조건을 비교합니다.

| 조건 | 프롬프트 |
|---|---|
| A. 기준 없음 | "영→한 번역가. 번역문만 출력" |
| B. 기준 명시 | A + 아래 3가지 기준 |

**B에서 명시한 기준과 자동 검사 방법**

| 기준 | 내용 | 검사 |
|---|---|---|
| 1. 용어집 | `LangChain`→`랭체인`, `developers`→`개발자분들` | 두 용어를 모두 사용하고 `LangChain`이 영어로 남지 않을 것 |
| 2. 말투 | 해요체 | 문장이 `-요`로 끝날 것 |
| 3. 길이 | 40자 이내 | 글자 수 ≤ 40 |

- **같은 조건**: 모델과 원문은 두 조건이 같고, 프롬프트의 기준 유무만 다릅니다.
- **5회 반복**: 모델의 응답은 실행마다 달라질 수 있어서 각 조건을 5회씩 실행하고, 기준별로 몇 번 지켜졌는지 셉니다.
- **예상 결과**: 기준이 없으면 `LangChain`은 영어로 남고 `-입니다` 체로 번역됩니다. 기준을 명시하면 **용어와 말투는 거의 항상 지켜집니다.** 반면 **글자 수 같은 수치 제약은 명시해도 지켜지지 않을 수 있습니다.** (이 모델에서 확인한 경향이며, 모델과 실행에 따라 다를 수 있습니다.)

> **정리**: 기준을 명시할수록 출력은 통제됩니다. 다만 모든 기준이 지켜지는 것은 아니므로, 중요한 기준(특히 글자 수 같은 수치 제약)은 **결과를 코드로 검증**해야 합니다.

In [ ]:
import re
from langchain_core.output_parsers import StrOutputParser

# [비교 실험] 번역 기준을 '명시하지 않을 때' vs '명시할 때'
# llm 은 위에서 만든 ChatUpstage 를 사용합니다. (모델·원문은 두 조건이 같고, 프롬프트의 기준 유무만 다름)

english_text = "LangChain is a framework that helps developers build AI applications with large language models."

# 조건 A: 기준 없이 번역만 요청
prompt_a = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a professional English-to-Korean translator. Output only the Korean translation."),
        ("human", "{english_text}"),
    ]
)

# 조건 B: 용어집·말투·길이 기준을 system 메시지에 명시
prompt_b = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a professional English-to-Korean translator. Output only the Korean translation.\n"
                   "Follow ALL of these rules:\n"
                   "1. Glossary: translate 'LangChain' as '랭체인' (never keep it in English), "
                   "and 'developers' as '개발자분들'.\n"
                   "2. Tone: polite 해요체 (every sentence ends with '-요').\n"
                   "3. Length: at most 40 Korean characters, including spaces."),
        ("human", "{english_text}"),
    ]
)

N = 5  # 각 조건을 반복하는 횟수 (모델 응답은 실행마다 달라질 수 있음)

# 기준별 자동 검사
checks = {
    "용어 (랭체인·개발자분들 사용)": lambda t: "랭체인" in t and "LangChain" not in t and "개발자분들" in t,
    "말투 (해요체 '-요')": lambda t: re.sub(r"[.!?\s]+$", "", t).endswith("요"),
    "길이 (40자 이내)": lambda t: len(t) <= 40,
}

# 두 조건을 각각 N번 실행
results = {}
for label, prompt in (("A. 기준 없음", prompt_a), ("B. 기준 명시", prompt_b)):
    chain = prompt | llm | StrOutputParser()
    results[label] = chain.batch([{"english_text": english_text}] * N, config={"max_concurrency": 3})

print(f"원문: {english_text}\n(각 조건 {N}회씩 실행)\n")
for label, outs in results.items():
    print(f"■ {label}")
    for o in outs:
        print(f"   - ({len(o)}자) {o}")
    print()

# 기준별 준수 횟수 비교
print("=" * 64)
print(f"{'기준':<30}{'A. 기준 없음':>14}{'B. 기준 명시':>14}")
for name, check in checks.items():
    a = sum(check(o) for o in results["A. 기준 없음"])
    b = sum(check(o) for o in results["B. 기준 명시"])
    print(f"{name:<30}{f'{a}/{N}':>14}{f'{b}/{N}':>14}")
print("=" * 64)

#### Groundness Check
* Groundedness Check API는 사용자가 제공한 Context(컨텍스트)에 대한 AI 어시스턴트의 응답이 실제로 그 컨텍스트에 기반하고 있는지 여부를 확인합니다.
* 최신버전에서는 UpstageGroundednessCheck 는 deprecated 되었음

In [ ]:
# from langchain_upstage import UpstageGroundednessCheck

# groundedness_check = UpstageGroundednessCheck()

# request_input = {
#     "context": "삼성전자는 연결 기준으로 매출 74.07조원, 영업이익 10.44조원의 2024년 2분기 실적을 발표했다. 전사 매출은 전분기 대비 3% 증가한 74.07조원을 기록했다. DS부문은 메모리 업황 회복으로 전분기 대비 23% 증가하고, SDC는 OLED 판매 호조로 증가했다.",
#     "answer": "삼성전자의 2024년 2분기 매출은 약 74.07조원이다.",
# }

# response = groundedness_check.invoke(request_input)
# print(response)  


#### UpstageEmbeddings

임베딩(Embedding)은 문자열을 **의미를 담은 숫자 벡터**로 바꾸는 것입니다. 벡터의 숫자 하나하나는 사람이 읽을 수 없지만, **벡터끼리의 거리(유사도)** 로 문장의 의미가 얼마나 가까운지 알 수 있습니다.

아래 예제는 다음 두 가지를 확인합니다.

**1) 문자열 ↔ 벡터 대응**
- `embed_documents(texts)`: 문자열 리스트를 벡터 리스트로 변환합니다. 입력 순서와 같은 순서로 반환됩니다.
- `embed_query(query)`: 질문 1개를 벡터 1개로 변환합니다.
- 모든 문자열은 길이와 상관없이 **같은 차원(4096)** 의 벡터가 됩니다. 앞 5개 값만 출력해 어떤 문자열의 벡터인지 대응을 확인합니다.

**2) 의미 확인 (코사인 유사도)**
- 질문 `What does Sung do?`의 벡터와 각 문서 벡터가 **같은 방향을 향하는 정도**를 계산합니다.

| 유사도 | 의미 |
|---|---|
| 1에 가까움 | 의미가 비슷함 |
| 0에 가까움 | 관련이 거의 없음 |
| 음수 | 반대 방향 |

- 결과에서 `Sung is a professor.`가 질문과 더 가깝게 나오면, 임베딩이 **단어가 같은지가 아니라 의미(Sung → 직업)** 를 반영했다는 뜻입니다. 이 원리로 RAG는 질문과 가까운 문서를 찾습니다.

In [ ]:
import math
from langchain_upstage import UpstageEmbeddings

# 임베딩 모델 생성 (solar-embedding-1-large: 문자열 1개를 4096개의 숫자로 변환)
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

# 임베딩할 문자열(문서)과 질문(쿼리)
texts = ["Sung is a professor.", "This is another document"]
query = "What does Sung do?"

# embed_documents(): 문자열 리스트 → 벡터 리스트 (입력 순서와 같은 순서로 반환)
#   doc_vectors[0] ← texts[0], doc_vectors[1] ← texts[1]
doc_vectors = embeddings.embed_documents(texts)

# embed_query(): 문자열 1개 → 벡터 1개 (검색용 질문을 임베딩할 때 사용)
query_vector = embeddings.embed_query(query)


def cosine_similarity(a, b):
    """두 벡터의 코사인 유사도를 반환합니다. (-1 ~ 1, 1에 가까울수록 의미가 비슷함)

    코사인 유사도 = 내적(a·b) / (|a| × |b|)
    → 벡터의 '길이'는 무시하고 '방향'이 얼마나 같은지만 비교합니다.
    """
    dot = sum(x * y for x, y in zip(a, b))        # 내적: 같은 위치의 값끼리 곱해서 모두 더함
    norm_a = math.sqrt(sum(x * x for x in a))     # 벡터 a의 길이
    norm_b = math.sqrt(sum(y * y for y in b))     # 벡터 b의 길이
    return dot / (norm_a * norm_b)


# ── 1) 문자열 ↔ 벡터 대응 확인 ──────────────────────────────
# 문자열 3개(문서 2개 + 질문 1개)와 벡터 3개를 zip()으로 한 쌍씩 묶어서 출력합니다.
#   - len(vec): 벡터의 차원 수 (모든 문자열이 같은 4096차원으로 변환됨)
#   - vec[:5] : 4096개 중 앞 5개만 표시 (전체를 출력하면 너무 길어서 일부만 확인)
#   - {v:+.4f}: 부호(+/-)를 붙여 소수점 4자리까지 표시
print("■ 1) 문자열 → 벡터 (앞 5개 값만 표시)")
for text, vec in zip(texts + [query], doc_vectors + [query_vector]):
    head = ", ".join(f"{v:+.4f}" for v in vec[:5])
    print(f"   {text!r:<28} → {len(vec)}차원 [{head}, ...]")

# ── 2) 의미 확인: 질문과 각 문서의 코사인 유사도 ────────────────
# 벡터의 숫자 자체는 읽을 수 없으므로, '질문 벡터와 얼마나 비슷한가'로 의미를 비교합니다.
#   - 질문 벡터 vs 문서 벡터를 하나씩 비교해 (문장, 유사도) 목록을 만듦
#   - 유사도가 높은 순으로 정렬해 출력 (RAG 검색에서 관련 문서를 찾는 것과 같은 원리)
#   - '█' 막대: 유사도 × 40 개 (값이 클수록 막대가 길어짐)
print(f"\n■ 2) 질문 {query!r} 와의 코사인 유사도")
scores = [(text, cosine_similarity(query_vector, vec)) for text, vec in zip(texts, doc_vectors)]
for text, score in sorted(scores, key=lambda x: x[1], reverse=True):
    print(f"   {score:.4f} {'█' * int(score * 40):<40} {text}")

# 유사도가 가장 큰 문장 = 질문과 의미가 가장 가까운 문장
best_text = max(scores, key=lambda x: x[1])[0]
print(f"\n   → 질문과 의미가 가장 가까운 문장: {best_text!r}")

#### UpstageDocumentParseLoader

PDF·이미지 같은 문서를 **레이아웃(제목, 문단, 표, 그림 등)을 인식해서** 텍스트로 변환하는 로더입니다. 단순 텍스트 추출(`PyPDFLoader` 등)보다 표와 구조가 잘 보존되어 RAG 전처리에 유용합니다.

**동작 흐름**: PDF 파일 → Upstage Document Parse API 호출 → LangChain `Document` 리스트 반환

**주요 파라미터**

| 파라미터 | 값 | 설명 |
|---|---|---|
| `split` | `"none"` / `"page"` / `"element"` | `Document`를 나누는 단위: 문서 전체 / 페이지별 / 요소별(제목·문단·표 등) |
| `output_format` | `"html"`(기본) / `"markdown"` / `"text"` | 본문 형식. 기본 html은 태그가 많아 읽기 어려워 markdown을 사용 |
| `coordinates` | `True`(기본) / `False` | 요소 좌표를 metadata에 포함할지 여부 |

**반환값 `Document`**
- `page_content`: 파싱된 본문
- `metadata`: 페이지 번호 등 부가 정보

> 파일을 파싱할 때마다 Upstage API가 호출됩니다. 같은 문서를 반복해서 쓸 때는 결과(`docs`)를 변수나 파일에 저장해 재사용하세요.

In [ ]:
from pathlib import Path
from langchain_upstage import UpstageDocumentParseLoader

# 파싱할 PDF 경로 (노트북 기준 상대경로)
PDF_PATH = Path("../data/tutorial-korean.pdf")
PREVIEW_PAGES = 3    # 내용을 미리 볼 페이지 수
PREVIEW_CHARS = 300  # 페이지당 미리 볼 글자 수

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {PDF_PATH.resolve()}")

# 로더 생성
#   - split="page"        : 페이지마다 Document 1개 생성 ("none": 문서 전체 1개, "element": 제목/문단/표 등 요소별 1개)
#   - output_format       : 파싱 결과 형식 ("html"(기본) / "markdown" / "text") → 읽기 쉬운 markdown 사용
#   - coordinates=False   : 요소의 좌표 정보를 metadata에 넣지 않음 (불필요한 정보 제거)
#   - API 키는 환경변수 UPSTAGE_API_KEY 에서 자동으로 읽음
loader = UpstageDocumentParseLoader(
    PDF_PATH,
    split="page",
    output_format="markdown",
    coordinates=False,
)

# load(): 모든 페이지를 한 번에 파싱 (Upstage API 호출 발생)
#   * 페이지가 많은 문서는 lazy_load()를 쓰면 페이지를 하나씩 처리해 메모리를 절약할 수 있음
docs = loader.load()

# 1) 결과 구조 확인: 페이지 수 = Document 수
print(f"파일: {PDF_PATH.name}")
print(f"Document 수(페이지 수): {len(docs)}")
print(f"Document 타입: {type(docs[0]).__name__}")  # Document: page_content(본문) + metadata(부가정보)

# 2) 앞 페이지 내용 미리보기: 본문(page_content)과 metadata를 함께 확인
for doc in docs[:PREVIEW_PAGES]:
    print("\n" + "=" * 60)
    print(f"metadata: {doc.metadata}")
    print(f"본문 길이: {len(doc.page_content)}자")
    print("-" * 60)
    print(doc.page_content[:PREVIEW_CHARS])